# Release - Automated Release Notes Agent

In this exercise, we build an agent that reads the raw commit history of a GitHub repository between two refs and produces polished, categorized release notes, ready to be published as a GitHub Release.

We use the official [GitHub MCP Server](https://github.com/github/github-mcp-server) to read commits and inspect pull requests.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GEMINI_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTHROPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


Additionally, this notebook expects a `GITHUB_PAT` entry in `.env` with `repo` scope.

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
github_mcp_url = "https://api.githubcopilot.com/mcp/"
github_token = os.environ["GITHUB_PAT"]

repo_owner = "VincentCoriou"
repo_name = "Bank-Account"

# Range of commits to turn into release notes.
branch_ref = "main"  # last released state
head_ref = "api"  # what we are about to ship

# The GitHub MCP server exposes ~50 tools, whose schemas (~13K tokens) are sent with every request. This does not
# fit in the free-tier input token budget of Gemma (16K/minute), so we only keep the tools needed for reading the history.
RELEASE_TOOL_NAMES = {"list_commits", "get_commit", "list_pull_requests", "pull_request_read", "list_tags", "list_releases"}

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
system_instructions_template_string = \
"""You are a technical writer specialized in translating engineering work into polished
release notes.

Given a range of commits on a GitHub repository, your goal is to produce a human-readable,
categorized release note, returned in Markdown as your final answer.

Workflow:
1. Use the GitHub MCP server to fetch every commit
   between `{{ branch_ref }}` and `{{ head_ref }}`.
2. For non-trivial commits, read titles, bodies,
   and/or diffs so you can explain *why* a change matters, not just *what* changed.
3. Group the commits into these sections (skip empty ones):
   - **Breaking changes** (flag clearly, with a migration hint where possible)
   - **New features**
   - **Improvements**
   - **Bug fixes**
   - **Internal / chores** (short)
4. For every bullet, write one concise, user-facing sentence. Link to the PR or commit SHA.

Keep the tone clear and confident. Avoid marketing fluff.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please produce release notes for {{ repo_owner }}/{{ repo_name }} covering commits from
`{{ branch_ref }}` to `{{ head_ref }}`."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render(
    repo_owner=repo_owner, repo_name=repo_name,
    branch_ref=branch_ref, head_ref=head_ref,
)
user_message = user_message_template.render(
    repo_owner=repo_owner, repo_name=repo_name,
    branch_ref=branch_ref, head_ref=head_ref,
)

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with tools_lib.mcp_session(github_mcp_url, authorization=github_token) as session:
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools if tool.name in RELEASE_TOOL_NAMES]
    done = False
    while not done:
        response = backend.generate(chat, tools=tools)
        done = True
        message = response.choices[0].message
        if message.content is not None:
            message.content = remove_thoughts(message.content)
        chat.append(message)
        for tool_call in message.tool_calls or ():
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)